In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

Oefening 1.1: Lees met pd.read_parquet het bestand ../data/uber_dataset_2025-01.parquet in. Deze bevat een (willekeurig gekozen) tiende van alle uber ritten in New York in Januari 2025. Bekijk het resulterende dataframe. Begrijp je de betekenis van alle kolommen?

In [ ]:
uber_df = ...

Oefening 1.2: Neem met onderstaande code een sample (steekproef) van 100 taxiritten. Wat zijn het gemiddelde en de mediaan van het aantal afgelegde mijlen? Waarom is de een hoger dan de ander? Draai dezelfde code nog een keer. Waarom geven we een seed als argument mee?

In [ ]:
seed = 42
df_sample = uber_df.sample(100, random_state = seed)                       
...

Oefening 1.3: Doe 100 verschillende (onhankelijke) steekproeven met weer 100 taxiritten elk, en herhaal bovenstaande. Je hebt nu 100 verschillende inschattingen van het gemiddelde en de mediaan. Plot deze om de spreiding hierin te zien. Interpreteer het resultaat. Waarom gebruiken we nu een random number generator (rng)?

In [ ]:
...

rng = np.random.default_rng(seed=42)

for _ in range(100):
    df_sample = uber_df.sample(100, random_state = rng)                     
    mean_results += [df_sample['trip_miles'].mean()]
    median_results += [df_sample['trip_miles'].median()]

fig, ax = plt.subplots()

sns.stripplot(data=[mean_results, median_results], jitter=0.1, size=5, ax=ax)

ax.set_xticks([0, 1])
ax.set_xticklabels(['mean', 'median'])

...

plt.show()

Oefening 1.4: Introduceer nu zelf outliers door te doen alsof voor een op de 300 ritjes de afstand per ongeluk in voet is ingevuld i.p.v. in mijl. Maak dezelfde plot als in de vorige opgave. Wat valt je op? Door welke eigenschappen van het gemiddelde / de mediaan komt dit? Wanneer zou je in de praktijk het gemiddelde willen gebruiken en wanneer de mediaan?

In [ ]:
# One in 300 rides has trip_miles reported in feet accidently
outlier_df = uber_df.assign(trip_miles = lambda df: df['trip_miles'].mask(
          df['ride_id'] % 300 == 0, 
          df['trip_miles'] * 5280)
      )

In [ ]:
...

Oefening 1.5: Neem een steekproef van 10 000 taxiritten. In welke 5 pickup_zones wordt het vaakst een fooi gegeven? Maak je als uber chauffeur in deze pickup zones dus een grote kans om een fooi te krijgen?

In [ ]:
rng = np.random.default_rng(seed = 42)

(
uber_df
    .sample(10000, random_state = rng)
    .assign(was_tipped = lambda df: (df['tips'] > 0))
    ...
     )

Oefening 1.6: Bereken nu voor deze vijf districten op de hele uber_df (dus, zonder eerst een steekproef te nemen) hoe vaak er fooi gegeven wordt. Hoe kan dit?

In [ ]:
...

Oefening 2.1: Voor welk deel van alle uberritten is er een fooi gegeven?

In [ ]:
...

Oefening 2.2: Neem nu een steekproef van 100 ritten. Voor welk deel van de ritten in deze steekproef is er een fooi gegeven? Gebruik onderstaande functie om op basis van deze steekproef een Confidence Interval te berekenen voor het deel van de ritten waarvoor een fooi wordt gegeven. 

Bevat deze de bij de vorige oefening berekende waarde? Hoe verandert de grootte van het interval als je meer ritten in de steekproef mee neemt?

In [ ]:
rng = np.random.default_rng(seed = 42)
sample_df = uber_df.sample(100, random_state = rng)

In [ ]:
...

In [ ]:
from statsmodels.stats.proportion import proportion_confint

def get_ci_for_fraction_of_rides_tipped(fraction_tipped: float, N: int, alpha = 0.05) -> Tuple[float, float]:
    """Use the Clopper-Pearson method to calculate a (1-alpha)% confidence interval for the fraction of rides where the driver received a tip.

    input:

    fraction_tipped: The fraction of rides in the sample where the driver received a tip.
    N: The number of rides in the sample

    returns:
    [CI lower limit, CI upper limit]
    """
    return proportion_confint(count=int(N*fraction_tipped), nobs=N, method='beta', alpha = alpha) 

In [ ]:
...

Bonus: Oefening 2.3: Als je bovenstaande herhaald voor 1000 onafhankelijke steekproeven (van 100 ritten elk), hoe vaak bevat de CI dan de echte (bij 1.1 berekende) waarde?

In [ ]:
rng = np.random.default_rng(seed = 42)

...

Bonus: Oefening 2.4 (Sample size estimation I): Wat zou de gemiddelde breedte van een 95%-CI voor een steekproef van 30 ritten zijn als het echte deel van de ritten waar een fooi zou worden gegeven 0.15 zou zijn? In welke context zou deze vraag relevant zijn?

In [ ]:
sample_size = 30
real_fraction_tipped = 0.15

rng = np.random.default_rng(42)
ci_widths = []
for _ in range(100): # The higher the more accurate

    number_tipped = rng.binomial(sample_size, real_fraction_tipped) # Randomly simulate whether the ride was tipped or not
    ...

...

Bonus: Oefening 2.5 (Sample size estimation II): Hoe groot moet de steekproefgrootte zijn om een gemiddelde CI-breedte van 0.2 te krijgen als het echte deel van de ritten waar een fooi wordt gegeven 0.15 is? En als dat 0.3 zou zijn? Hoe zou je deze berekening in de praktijk gebruiken?

In [ ]:
real_fraction_tipped = 0.15

...


Bonus: Oefening 2.6: Draai onderstaande code om een Bayesiaanse credible interval te bereken met als prior verdeling Beta(alpha = 0.5, beta = 0.5). Vervang deze prior door je eigen prior verdeling (dwz. welke verdeling van kansen je van te voren zou hebben ingeschat) met bijvoorbeeld pm.Uniform, pm.Normal, pm.CustomDist (mooier maar moeilijker), of door te parameters van de beta verdeling aan te passen. Hoe verandert de confidence interval? Kun je er met een andere prior voor zorgen dat deze heel nauw, of juist heel breed is? Of dat deze ver weg van de echte parameter ligt?

In [ ]:
rng = np.random.default_rng(seed = 42)
num_samples = 100
sample_df = uber_df.sample(num_samples, random_state = rng)

In [ ]:
import pymc as pm


num_successes = (
    sample_df
    .assign(was_tipped = lambda df: (df['tips'] > 0))
    .agg(fraction_tipped = ('was_tipped', 'sum'))
    .iloc[0,0]
)


with pm.Model() as model:
    p = pm.Beta("p", alpha=0.5, beta=0.5) # Prior distribution
    pm.Binomial("obs", n=num_samples, p=p, observed=num_successes)
    idata = pm.sample(random_seed=rng) # Samples from posterior distribution


hdi = pm.stats.hdi(idata, prob=0.95)["p"].values # 'Highest density interval': Smallest 95% interval
print(f"95% credible interval: [{hdi[0]}, {hdi[1]}]")

### Bootstrap CI

In [ ]:
from scipy.stats import bootstrap, pearsonr, spearmanr 

Oefening 2.7: Neem een steekproef van 1000 ritten uit uber_df en gebruik scipy.stats.bootstrap om een CI voor de mean en de median van trip_miles te berekenen. Bevat deze het echte gemiddelde en de echte mediaan van de volledige uber_df?

In [ ]:
sample_size = 1000
rng = np.random.default_rng(seed = 42)


...

Oefening 2.8: Herhaal oefening 2.7 voor de Pearson en Spearman correlaties tussen trip_miles and driver pay (i.p.v. de mean en median), met steekproefgrootte van 100 ritten.

In [ ]:
sample_size = 100
rng = np.random.default_rng(seed = 42)

print(f"Real Pearson correlation = {pearsonr(uber_df['trip_miles'], uber_df['driver_pay']).statistic}, real Spearman correlation = {spearmanr(uber_df['trip_miles'], uber_df['driver_pay']).statistic}")

df_sample = uber_df.sample(sample_size, random_state = rng)
data = [df_sample['trip_miles'].to_numpy(), df_sample['driver_pay'].to_numpy()]

# nb: scipy.stats.bootstrap will require the argument 'paired = True'! Why?

...

Bonus: Oefening 2.9: Maak een 'model' dat om te verspellen of er fooi wordt gegeven willekeurig ja of nee gokt. Kies je favoriete metric, bijvoorbeeld sklearn.metrics.accuracy_score. Maak een 'testset' door een steekproef uit uber_df te maken. Hoe goed scoort je algorithme op deze testset? En op een nieuwe, onafhankelijk gekozen testset? Gebruik de Bootstrap om een CI te berekenen.

Bonus: Oefening 2.10: Doe hetzelfde als voor oefening 2.7 voor 300 steekproeven van 50 ritten elk, en bekijk hoe vaak de Bootstrap CI de echte mean en median bevat. Herhaal dit voor steekproeven van 20 ritten elk. Wat valt je op?

In [ ]:
rng = np.random.default_rng(seed = 42)

sample_size = 20

real_mean = uber_df['trip_miles'].mean()
real_median = np.median(uber_df['trip_miles'].to_numpy())

real_mean_in_ci = []
real_median_in_ci = []

for it in range(300):
    if it%10 == 0:
        print(f"Processed {it} samples.")

    df_sample = uber_df.sample(sample_size, random_state =rng)

    data = [df_sample['trip_miles'].to_numpy()]

    mean_ci = bootstrap(data, lambda x: x.mean(), confidence_level=0.95, random_state = rng).confidence_interval
    median_ci = bootstrap(data, np.median, confidence_level=0.95, random_state = rng).confidence_interval

    real_mean_in_ci += [(mean_ci[0] <= real_mean <= mean_ci[1])]
    real_median_in_ci += [(median_ci[0] <= real_median <= median_ci[1])]

print(sum(real_mean_in_ci) / len(real_mean_in_ci))
print(sum(real_median_in_ci) / len(real_median_in_ci))

Bonus: Oefening 2.11: Herhaal (een deel van) oefeningen 2.7-2.10 met een zelfgeschreven implementatie van de bootstrap i.p.v. die van scipy.